In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import numpy as np

# Thêm thư mục gốc vào path để import từ src
sys.path.append(os.path.abspath(".."))
from src.data_csp import load_pilot_data, solve_csp

print("Đã nạp module thành công!")

In [ ]:
import os
import numpy as np

os.makedirs("../data", exist_ok=True)

num_samples = 15000
n_size = 6
seq_len = n_size * n_size

# Bảng Sudoku 6x6 chuẩn hợp lệ (6 khối con 2x3)
base_solution = np.array([
    [1, 2, 3, 4, 5, 6],
    [4, 5, 6, 1, 2, 3],
    [2, 3, 1, 5, 6, 4],
    [5, 6, 4, 2, 3, 1],
    [3, 1, 2, 6, 4, 5],
    [6, 4, 5, 3, 1, 2]
], dtype=np.int64)

quizzes = []
solutions = []

print(f"Đang sinh {num_samples} bảng Sudoku 6x6 hợp lệ...")

for idx in range(num_samples):
    sol = base_solution.copy()
    
    # 1. Hoán đổi ngẫu nhiên các chữ số từ 1..6
    perm = np.random.permutation(np.arange(1, 7))
    mapping = {i + 1: perm[i] for i in range(6)}
    sol = np.vectorize(mapping.get)(sol)
    
    # 2. Xáo trộn vị trí các hàng trong cùng một khối 2 hàng
    for block_r in range(0, 6, 2):
        if np.random.rand() > 0.5:
            sol[[block_r, block_r + 1]] = sol[[block_r + 1, block_r]]
            
    # 3. Xáo trộn vị trí các cột trong cùng một khối 3 cột
    for block_c in range(0, 6, 3):
        col_perm = block_c + np.random.permutation(3)
        sol[:, block_c:block_c + 3] = sol[:, col_perm]
        
    sol_1d = sol.flatten().astype(np.int64)
    
    # 4. Đục lỗ (mask) ngẫu nhiên từ 14 đến 22 ô trống (giá trị 0)
    quiz_1d = sol_1d.copy()
    num_missing = np.random.randint(14, 23)
    mask_indices = np.random.choice(seq_len, size=num_missing, replace=False)
    quiz_1d[mask_indices] = 0
    
    quizzes.append(quiz_1d)
    solutions.append(sol_1d)
    
    if (idx + 1) % 5000 == 0 or (idx + 1) == num_samples:
        print(f"-> Đã sinh: {idx + 1}/{num_samples} mẫu...")

quizzes = np.array(quizzes, dtype=np.int64)
solutions = np.array(solutions, dtype=np.int64)

# Lưu dữ liệu theo Data Contract
data_path = "../data/sudoku_pilot.npz"
np.savez_compressed(data_path, quizzes=quizzes, solutions=solutions)

print(f"-> Đã tạo thành công dữ liệu tại: {data_path}")
print(f"-> Shape quizzes: {quizzes.shape}, dtype: {quizzes.dtype}")
print(f"-> Shape solutions: {solutions.shape}, dtype: {solutions.dtype}")

In [ ]:
# Lấy 1 mẫu test để chạy thử giải thuật CSP
quiz_test = quizzes[0]

# Chạy giải đố bằng MRV với bảng 6x6
pred_board, step_history = solve_csp(quiz_test, method="mrv", n_size=6)

print("Số ô trống còn lại:", (pred_board == 0).sum())
print("Tổng số bước giải:", len(step_history))

if len(step_history) > 0:
    first_step = step_history[0]
    print("\nCác khóa trong step_history[0]:", list(first_step.keys()))
    assert list(first_step.keys()) == ['step', 'unmasked_indices', 'values', 'margins', 'board_state']
    print("=> HỢP ĐỒNG DỮ LIỆU (DATA CONTRACT) CHUẨN XÁC!")

In [ ]:
# Lấy 200 mẫu test cuối để chạy đánh giá baseline
test_quizzes = quizzes[-200:]
preds = []

start_time = time.time()
for q in test_quizzes:
    pred, _ = solve_csp(q, method="mrv", n_size=6)
    preds.append(pred)
elapsed_time = time.time() - start_time

print(f"Thời gian giải {len(test_quizzes)} mẫu test: {elapsed_time:.4f} giây")

# Lưu file kết quả baseline cho TV4
result_path = "../baseline_csp_results.npz"
np.savez_compressed(
    result_path,
    preds=np.array(preds),
    time_taken=elapsed_time
)
print(f"Đã xuất kết quả baseline tại: {result_path}")

In [ ]:
import os
import time
import pickle
import numpy as np

from src.data_csp import load_pilot_data, solve_csp
from src.interfaces import validate_step_history

# 1. Cấu hình đường dẫn thư mục lưu trữ
# Nếu chạy trên Colab mount Drive:
# SHARED_DIR = "/content/drive/MyDrive/IntroAI_Nhom5_Shared"
# Nếu chạy cục bộ trên máy tại thư mục dự án:
# Tự động lùi về thư mục gốc nếu notebook đang chạy trong thư mục con "notebooks/"
SHARED_DIR = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."

data_path = os.path.join(SHARED_DIR, "data", "sudoku_pilot.npz")
results_dir = os.path.join(SHARED_DIR, "results")
os.makedirs(results_dir, exist_ok=True)

# 2. Đọc dữ liệu và cố định 200 bảng cuối làm tập Test
quizzes, solutions = load_pilot_data(data_path)
test_inputs = quizzes[-200:].astype(np.int64)
test_targets = solutions[-200:].astype(np.int64)

seq_len = test_inputs.shape[1]
n_size = int(np.sqrt(seq_len))

print(f"-> Đã nạp tập test: {test_inputs.shape}, kích thước n_size = {n_size}")

# 3. Chạy thực nghiệm 3 thuật toán CSP
methods = ["backtracking", "mrv", "ac3"]
preds_dict = {}
histories_dict = {}
solve_times = []

for m in methods:
    print(f"\n--- Đang chạy phương pháp CSP: {m.upper()} ---")
    preds_list, hist_list = [], []
    t0 = time.time()
    
    for i in range(len(test_inputs)):
        pred_1d, hist = solve_csp(test_inputs[i], method=m, n_size=n_size)
        
        # Kiểm tra tính hợp lệ của step_history theo Data Contract
        assert validate_step_history(hist, seq_len=seq_len), f"Sai Data Contract ở {m} tại mẫu {i}!"
        
        preds_list.append(pred_1d)
        hist_list.append(hist)
        
        if (i + 1) % 50 == 0 or (i + 1) == len(test_inputs):
            print(f"Đã giải {i + 1}/{len(test_inputs)} bảng...")
            
    avg_time = (time.time() - t0) / len(test_inputs)
    preds_dict[m] = np.array(preds_list, dtype=np.int64)  # Shape (200, L)
    histories_dict[m] = hist_list
    solve_times.append(avg_time)
    print(f"-> {m.upper()} hoàn thành: thời gian trung bình {avg_time:.4f}s / bảng")

# 4. Xuất file .npz chứa kết quả dự đoán và thời gian chạy cho TV4
results_npz_path = os.path.join(results_dir, "baseline_csp_results.npz")
np.savez(
    results_npz_path,
    inputs=test_inputs,
    targets=test_targets,
    preds_backtracking=preds_dict["backtracking"],
    preds_mrv=preds_dict["mrv"],
    preds_ac3=preds_dict["ac3"],
    solve_times=np.array(solve_times, dtype=np.float64)  # [bt_time, mrv_time, ac3_time]
)
print(f"\n[TV1 PASS] Đã lưu file kết quả: {results_npz_path}")

# 5. Xuất file .pkl chứa lịch sử các bước giải cho TV4 và TV5
histories_pkl_path = os.path.join(results_dir, "baseline_csp_histories.pkl")
with open(histories_pkl_path, "wb") as f:
    pickle.dump(histories_dict, f)
print(f"[TV1 PASS] Đã lưu file lịch sử: {histories_pkl_path}")